In [46]:
import pandas as pd
import re
import unicodedata

In [47]:
df = pd.read_csv("../dados/clientes_distribuidora_vetor.csv",
                 sep=";", dtype=str, keep_default_na=False)

In [48]:
df.shape   # deve mostrar (47, 11)

(47, 11)

In [49]:
# para recarregar automaticamente as funções que vocês forem movendo para distribuidora/etl/:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
df.head()

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,28/02/2022,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,"Dec 24, 2023",Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,08-01-2022,Diego Martins


In [51]:
df.tail()

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
42,CLI0043,Bazar Sol Nascente,43.671.369/5944-06,61911086890,contato@bazarsolnasc.com.br,Águas Claras Norte,71901-010,Brasília,DF,"nov 19, 2021",Diego Martins
43,CLI0044,Depósito Cerrado Sul,53394210470952,(61) 3402-6352,contato@depósitocerr.com.br,Águas Claras/DF,71900-100,Brasília,DF,2022-07-24,Camila Torres
44,CLI0045,Loja Multi Utilidades,47.451.712/3685-16,(61) 3370-9791,contato@lojamultiuti.com.br,Aguas claras,71900-100,Brasília,DF,04/08/2023,Bruno Almeida
45,CLI0046,Armazém Boa Colheita,37.098.593/1746-12,(61) 3138-5986,contato@armazémboaco.com.br,Guará II,,Brasília,DF,2021-02-16,Camila Torres
46,CLI0047,Mercadinho Sem Endereço Completo,82.675.869/2617-96,61914391105,contato@semendereco.com.br,,,Brasília,DF,2022-06-23,Juliana Reis


In [52]:
# Obter informações gerais (tipos de dados, valores não nulos)
print("\nInformações gerais do dataset")
df.info()

# Verificar as dimensões (shape)
print(f"\n Número total de registros(len) = {len(df)}, cada linha de registro é um cliente")

print("\n--- Dimensões do dataset da distribuidora Vetor ---")
print(f"O dataset possui: {df.shape[0]} linhas/clientes e {df.shape[1]} colunas, de 0 a 10")





Informações gerais do dataset
<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      47 non-null     str  
 6   cep                   47 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
dtypes: str(11)
memory usage: 4.2 KB

 Número total de registros(len) = 47, cada linha de registro é um cliente

--- Dimensões do dataset da distribuidora Vetor ---
O dataset possui: 47 linhas/clientes e 11 colunas, de 0 a 10


In [53]:
display(df)

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,28/02/2022,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,"Dec 24, 2023",Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,08-01-2022,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82.278.248/9638-34,+55 61 97118-8177,contato@casadastinta.com.br,Águas Claras,,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83.930.103/1051-83,61999788677,contato@distribuidor.com.br,TAGUATINGA,,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31.165.667/0106-51,(61) 94139-4116,contato@bazarestrela.com.br,V. Pires,72006-400,Brasília,DF,"Aug 18, 2022",Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10.801.326/7736-02,+55 61 91035-7396,contato@supermercado.com.br,Aguas claras,71926-750,Brasília,DF,2023-08-09,Juliana Reis
9,CLI0010,Papelaria Horizonte,34.309.805/0097-88,(61) 91931-9320,contato@papelariahor.com.br,Águas Claras/DF,71950-000,Brasília,DF,03/03/2021,Diego Martins


## T - Transformação

Observações 
- todas as colunas estão como tipo de texto, inclusive as númericas, valores faltando em várias colunas, falta de padronização no cnpj, telefone, bairro, cep, data de cadastro, os valores vazios estão sendo contados no total

In [54]:
#Corrigir os valores faltantes nas colunas como null, 
# Valores preenchidos apenas com espaços em branco ou strings vazias ("") foram convertidos para o nulo nativo do Pandas (pd.NA).
# antes eles estavam sendo contados como um texto vázio, agora não nulos, não tem nada escrito
df = df.replace(r'^\s*$', pd.NA, regex=True)

# o cep tem valores 0000000 que também devem ser mudados para nulos, pois não tem sentido nenhum

In [55]:

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      46 non-null     str  
 6   cep                   34 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
dtypes: str(11)
memory usage: 4.2 KB


- Agora pode notar que tem valores nulos no bairro informado e no cep, pois o total não bateu com o total de 47 registros

In [56]:
# A função len() conta a quantidade total de linhas (registros) existentes no DataFrame df e guarda esse número inteiro na variável total
total = len(df)
# df.isna() varre a tabela inteira e transforma os dados em respostas de "Sim ou Não" (True para o que for nulo/pd.NA, e False para o que estiver preenchido
# .sum() soma essas respostas (onde True vale 1 e False vale 0) coluna por coluna. 
# O resultado armazenado na variável nulos é uma lista (Series do Pandas) contendo o nome de todas as colunas e a quantidade exata de nulos em cada uma delas.
nulos = df.isna().sum()

# Pega a lista completa gerada na etapa anterior e aplica uma máscara de filtro (> 0). 
# criando uma nova lista contendo apenas as colunas que possuem pelo menos 1 dado nulo, descartando as colunas que estão 100% preenchidas.
colunas_com_nulos = nulos[nulos > 0]


# Percorre as colunas filtradas e imprime a frase no formato solicitado
# o método .items() pega a lista filtrada e separa as informações em pares: o índice (que é o nome da coluna) e o valor (que é a quantidade de nulos).
# O laço for estabelece a regra: "Para cada par dessa lista, guarde o nome na variável coluna e o número na variável qtd, e execute a linha de baixo".
for coluna, qtd in colunas_com_nulos.items():
    print(f"Na coluna {coluna} existe {qtd} registros Nulos/NaN/Sem preenchimento de um total de {total}")

Na coluna bairro_informado existe 1 registros Nulos/NaN/Sem preenchimento de um total de 47
Na coluna cep existe 13 registros Nulos/NaN/Sem preenchimento de um total de 47


- na coluna cep tem valores nulos como 00000 que precisa ser registrado como inválido/nulo NaN, pois não tem valor prático o cep 0000000 e está sendo contado como valor válido

In [57]:
# 1. Ajustando a função do CEP para retornar a tupla (Valor, Critério)
def normalizar_cep_auditoria(valor):
    # Se já for nulo (NaN, pd.NA, None)
    if pd.isna(valor):
        return pd.NA, 'Já era Nulo original'
    
    # Extrai apenas os números do texto
    apenas_numeros = re.sub(r'\D', '', str(valor))
    
    # Se, ao remover as letras/traços, não sobrar nenhum número
    if len(apenas_numeros) == 0:
        return pd.NA, 'Inválido: Sem caracteres numéricos'
        
    # Se o número tiver mais de 8 dígitos (erro de digitação grave)
    if len(apenas_numeros) > 8:
        return pd.NA, f'Inválido: Excesso de dígitos ({len(apenas_numeros)} dígitos)'
    
    # Aplica o preenchimento de zeros à esquerda se tiver 8 ou menos dígitos
    cep_formatado = apenas_numeros.zfill(8)
        
    # Validação de preenchimento "preguiçoso"
    if cep_formatado == '00000000':
        return pd.NA, 'Inválido: Preenchimento genérico (00000000)'
        
    # Se passou por todas as validações, o CEP é válido.
    # Vamos registrar na auditoria se ele precisou de zeros à esquerda ou se já estava com 8 dígitos.
    if len(apenas_numeros) < 8:
        return cep_formatado, 'Válido: Higienizado e zeros à esquerda restaurados'
    else:
        return cep_formatado, 'Válido: Higienizado (tamanho original correto)'

# 2. Recebendo a tupla em DUAS colunas simultaneamente
# O apply varre a coluna antiga 'cep', roda a função, e o pd.Series separa o resultado.
# O dado limpo vai para a coluna 'cep' e a mensagem vai para a nova 'auditoria_cep'.
df[['cep', 'auditoria_cep']] = df['cep'].apply(
    lambda x: pd.Series(normalizar_cep_auditoria(x))
)

# 3. Contagem rápida para verificar o que aconteceu (Opcional, apenas para log visual)
print("\n--- Resumo da Auditoria do CEP ---")
print(df['auditoria_cep'].value_counts().to_string())


--- Resumo da Auditoria do CEP ---
auditoria_cep
Válido: Higienizado (tamanho original correto)    31
Já era Nulo original                              13
Inválido: Preenchimento genérico (00000000)        3


- colocamos os cep 00000 como nulos, agora existem apenas 31 válidos

In [58]:

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      46 non-null     str  
 6   cep                   31 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
 11  auditoria_cep         47 non-null     str  
dtypes: str(12)
memory usage: 4.5 KB


In [59]:
df.head()

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel,auditoria_cep
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926750,Brasília,DF,2022-06-21,Bruno Almeida,Válido: Higienizado (tamanho original correto)
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140010,Brasília,DF,2023-10-07,Camila Torres,Válido: Higienizado (tamanho original correto)
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140010,Brasília,DF,28/02/2022,Camila Torres,Válido: Higienizado (tamanho original correto)
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929000,Brasília,DF,"Dec 24, 2023",Diego Martins,Válido: Higienizado (tamanho original correto)
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,NaN,Brasília,DF,08-01-2022,Diego Martins,Inválido: Preenchimento genérico (00000000)


In [60]:
# 1. Função para normalizar o CNPJ e gerar o log de auditoria
def normalizar_cnpj_auditoria(valor):
    # Se o valor original for nulo ou apenas espaços em branco
    if pd.isna(valor) or str(valor).strip() == '':
        return pd.NA, 'Já era Nulo ou Vazio'
    
    # Extrai apenas os números, removendo qualquer formatação prévia (pontos, barras, traços)
    apenas_numeros = re.sub(r'\D', '', str(valor))
    
    # Se, após a limpeza, não sobrar nenhum número
    if len(apenas_numeros) == 0:
        return pd.NA, 'Inválido: Sem caracteres numéricos'
        
    # Um CNPJ válido no Brasil possui exatamente 14 dígitos
    if len(apenas_numeros) == 14:
        # Reconstrói a string aplicando a máscara oficial (XX.XXX.XXX/XXXX-XX)
        cnpj_formatado = f"{apenas_numeros[:2]}.{apenas_numeros[2:5]}.{apenas_numeros[5:8]}/{apenas_numeros[8:12]}-{apenas_numeros[12:]}"
        return cnpj_formatado, 'Válido: Higienizado e máscara aplicada'
        
    # Se não tiver 14 dígitos (pode ser um CPF inserido por engano ou erro de digitação),
    # devolvemos o valor original sujo para não perder a informação e apontamos o erro.
    return valor, f'Inválido/Suspeito: Possui {len(apenas_numeros)} dígitos numéricos (esperado: 14)'

# 2. Recebendo a tupla em DUAS colunas simultaneamente
# A coluna 'cnpj' recebe o dado formatado e a nova coluna 'auditoria_cnpj' recebe o log
df[['cnpj', 'auditoria_cnpj']] = df['cnpj'].apply(
    lambda x: pd.Series(normalizar_cnpj_auditoria(x))
)

# 3. Contagem rápida para verificar os logs (visualização no Jupyter)
print("\n--- Resumo da Auditoria do CNPJ ---")
print(df['auditoria_cnpj'].value_counts().to_string())


--- Resumo da Auditoria do CNPJ ---
auditoria_cnpj
Válido: Higienizado e máscara aplicada    47


In [61]:
print(df['cnpj'].to_string())

0     84.959.310/3413-16
1     28.327.648/3503-05
2     42.388.496/9653-28
3     69.784.801/8451-46
4     89.325.288/0957-01
5     82.278.248/9638-34
6     83.930.103/1051-83
7     31.165.667/0106-51
8     10.801.326/7736-02
9     34.309.805/0097-88
10    93.990.916/9985-43
11    79.911.838/4251-35
12    24.118.244/9353-48
13    27.868.011/2805-98
14    69.232.260/2563-42
15    36.541.458/6850-14
16    93.406.088/3561-59
17    82.366.299/4680-44
18    48.951.343/3200-37
19    20.163.287/0831-72
20    77.434.873/4714-34
21    66.587.603/6690-96
22    34.670.656/2729-80
23    37.556.464/1708-05
24    27.193.745/2991-24
25    49.190.586/5185-06
26    87.769.453/1473-79
27    49.480.831/3678-37
28    85.685.574/4431-35
29    34.352.408/2400-84
30    04.711.671/9022-94
31    74.964.990/9133-41
32    03.447.134/9361-83
33    17.464.887/7190-65
34    02.787.429/6717-56
35    07.154.516/8087-60
36    10.932.480/8613-17
37    78.263.982/1465-84
38    58.867.533/9636-05
39    71.870.262/1745-96


In [62]:
# 1. Função para normalizar o Telefone e gerar o log de auditoria
def normalizar_telefone_auditoria(valor):
    # Se o valor original for nulo ou apenas espaços em branco
    if pd.isna(valor) or str(valor).strip() == '':
        return pd.NA, 'Já era Nulo ou Vazio'
    
    # Extrai apenas os números, removendo parênteses, traços, espaços ou o sinal de '+'
    numeros = re.sub(r'\D', '', str(valor))
    
    # Se, após a limpeza, não sobrar nenhum número
    if len(numeros) == 0:
        return pd.NA, 'Inválido: Sem caracteres numéricos'
        
    # Flag para registar se removemos o código do país (+55)
    criterio_extra = ""
    if numeros.startswith('55') and len(numeros) in [12, 13]:
        numeros = numeros[2:] # Corta o '55' do início
        criterio_extra = " (DDI +55 removido)"
        
    # Regra Celular: 11 dígitos (DDD + 9 dígitos)
    if len(numeros) == 11:
        telefone_formatado = f"{numeros[:2]} {numeros[2:7]}-{numeros[7:]}"
        return telefone_formatado, f'Válido: Máscara Celular aplicada{criterio_extra}'
        
    # Regra Fixo: 10 dígitos (DDD + 8 dígitos)
    elif len(numeros) == 10:
        telefone_formatado = f"{numeros[:2]} {numeros[2:6]}-{numeros[6:]}"
        return telefone_formatado, f'Válido: Máscara Fixo aplicada{criterio_extra}'
        
    # Se não tiver 10 nem 11 dígitos, devolvemos o valor limpo (para não perder a informação)
    # e apontamos o erro na auditoria para revisão manual posterior.
    return numeros, f'Inválido/Suspeito: Possui {len(numeros)} dígitos numéricos (esperado: 10 ou 11)'

# 2. Recebendo a tupla em DUAS colunas simultaneamente
# A coluna 'telefone' recebe o dado formatado e a nova 'auditoria_telefone' recebe o log
df[['telefone', 'auditoria_telefone']] = df['telefone'].apply(
    lambda x: pd.Series(normalizar_telefone_auditoria(x))
)

# 3. Contagem rápida para verificar os logs
print("\n--- Resumo da Auditoria do Telefone ---")
print(df['auditoria_telefone'].value_counts().to_string())


--- Resumo da Auditoria do Telefone ---
auditoria_telefone
Válido: Máscara Celular aplicada                       21
Válido: Máscara Fixo aplicada                          14
Válido: Máscara Celular aplicada (DDI +55 removido)    12


In [63]:
print(df['telefone'].to_string())

0     61 97085-5700
1     61 95386-2084
2     61 92489-1771
3     61 98433-1053
4     61 95127-3847
5     61 97118-8177
6     61 99978-8677
7     61 94139-4116
8     61 91035-7396
9     61 91931-9320
10    61 94203-5886
11    61 93117-2421
12    61 95114-1832
13    61 98041-3088
14     61 3491-1634
15    61 95349-1626
16    61 99903-8359
17    61 98043-6279
18     61 3970-2193
19    61 99702-6198
20    61 95291-0691
21    61 91096-6763
22    61 91444-2375
23    61 92124-1659
24     61 3692-1420
25    61 92729-8102
26     61 3784-7209
27    61 98342-1282
28     61 3860-9779
29    61 99042-2680
30     61 3354-2940
31    61 94929-3847
32    61 95352-3330
33     61 3356-1423
34    61 95309-1709
35    61 99494-6927
36     61 3763-3522
37     61 3389-5891
38    61 97246-7325
39     61 3972-6383
40     61 3400-6873
41    61 95712-9955
42    61 91108-6890
43     61 3402-6352
44     61 3370-9791
45     61 3138-5986
46    61 91439-1105


In [64]:
# 1. Função para normalizar o E-mail e gerar o registo de auditoria
def normalizar_email_auditoria(valor):
    # Se o valor original for nulo ou apenas espaços em branco
    if pd.isna(valor) or str(valor).strip() == '':
        return pd.NA, 'Já era Nulo ou Vazio'
    
    # Remove espaços nas pontas e guarda o original para comparação
    texto_original = str(valor).strip()
    
    # Converte tudo para minúsculas
    texto_limpo = texto_original.lower()
    
    # Remove os acentos (ex: transformando 'joão@email.com' em 'joao@email.com')
    texto_sem_acento = unicodedata.normalize('NFKD', texto_limpo).encode('ASCII', 'ignore').decode('utf-8')
    
    # Validação Básica: Um e-mail minimamente válido precisa ter '@' e pelo menos um ponto '.'
    if '@' not in texto_sem_acento or '.' not in texto_sem_acento:
        # Devolve o texto limpo, mas assinala para revisão
        return texto_sem_acento, 'Inválido/Suspeito: Não possui formato padrão de e-mail (@ e .)'
        
    # Verifica se houve necessidade de alterar o texto original
    if str(valor) != texto_sem_acento:
        return texto_sem_acento, 'Válido: Higienizado (espaços, minúsculas e/ou acentos corrigidos)'
    
    # Se passou por tudo e não precisou de correção
    return texto_sem_acento, 'Válido: Já estava no padrão correto'

# 2. Recebendo o tuplo em DUAS colunas simultaneamente
# A coluna 'email' recebe o dado formatado e a nova 'auditoria_email' recebe o log
df[['email', 'auditoria_email']] = df['email'].apply(
    lambda x: pd.Series(normalizar_email_auditoria(x))
)

# 3. Contagem rápida para verificar os registos de auditoria
print("\n--- Resumo da Auditoria de E-mail ---")
print(df['auditoria_email'].value_counts().to_string())


--- Resumo da Auditoria de E-mail ---
auditoria_email
Válido: Já estava no padrão correto                                  37
Válido: Higienizado (espaços, minúsculas e/ou acentos corrigidos)    10


In [65]:
# ordena e imprime linha por linha
bairros_ordenados = df['bairro_informado'].unique()
bairros_ordenados.sort()

for bairro in bairros_ordenados:
    print(bairro)

AGUAS CLARAS
Aguas Claras - DF
Aguas claras
Guará I
Guará II
Norte (Aguas Claras)
Norte (Águas Claras)
Sul (Aguas Claras)
Sul (Águas Claras)
TAGUATINGA
Taguatinga
Taguatinga - DF
Taguatinga/DF
V. Pires
VICENTE PIRES
Vicente Pires
Vicente Pires - DF
Vte Pires
aguas claras
guara - df
taguatinga sul
Águas Claras
Águas Claras Norte
Águas Claras Sul
Águas Claras/DF
nan


In [66]:
# 1. Função para normalizar o Bairro e gerar o log de auditoria
def normalizar_bairro_auditoria(valor):
    # Se for nulo ou vazio
    if pd.isna(valor) or str(valor).strip() == '':
        return pd.NA, 'Já era Nulo ou Vazio'
    
    texto_original = str(valor).strip()
    texto_upper = texto_original.upper()
    
    # Variável para guardar o resultado da padronização
    bairro_padrao = None
    
    # Regra do Vicente Pires
    if 'PIRES' in texto_upper:
        bairro_padrao = 'Vicente Pires'
        
    # Regras de Águas Claras (Norte, Sul ou Padrão)
    elif 'AGUAS' in texto_upper or 'ÁGUAS' in texto_upper:
        if 'NORTE' in texto_upper:
            bairro_padrao = 'Águas Claras Norte'
        elif 'SUL' in texto_upper:
            bairro_padrao = 'Águas Claras Sul'
        else:
            bairro_padrao = 'Águas Claras'
            
    # Regras do Guará (I, II ou Padrão)
    elif 'GUARA' in texto_upper or 'GUARÁ' in texto_upper:
        if 'II' in texto_upper:
            bairro_padrao = 'Guará II'
        elif 'I' in texto_upper:
            bairro_padrao = 'Guará I'
        else:
            bairro_padrao = 'Guará'
            
    # Regras de Taguatinga (Sul ou Padrão)
    elif 'TAGUATINGA' in texto_upper:
        if 'SUL' in texto_upper:
            bairro_padrao = 'Taguatinga Sul'
        else:
            bairro_padrao = 'Taguatinga'
            
    # Se não caiu em nenhuma regra de canonização
    if bairro_padrao is None:
        return texto_original, 'Sem regra de canonização: Mantido o valor original limpo'
        
    # Verifica se o texto original já estava digitado perfeitamente
    if texto_original == bairro_padrao:
        return bairro_padrao, 'Válido: Já estava no padrão correto'
    else:
        # Mostra exatamente o que foi transformado
        return bairro_padrao, f'Canonizado: Transformado de "{texto_original}" para "{bairro_padrao}"'

# 2. Recebendo a tupla em DUAS colunas simultaneamente
# A coluna original recebe o bairro padronizado e a nova recebe o log
df[['bairro_informado', 'auditoria_bairro']] = df['bairro_informado'].apply(
    lambda x: pd.Series(normalizar_bairro_auditoria(x))
)

# 3. Contagem rápida para verificar os logs
print("\n--- Resumo da Auditoria de Bairros ---")
print(df['auditoria_bairro'].value_counts().to_string())


--- Resumo da Auditoria de Bairros ---
auditoria_bairro
Válido: Já estava no padrão correto                                             13
Canonizado: Transformado de "Aguas claras" para "Águas Claras"                   4
Canonizado: Transformado de "Taguatinga - DF" para "Taguatinga"                  3
Canonizado: Transformado de "aguas claras" para "Águas Claras"                   3
Canonizado: Transformado de "Vte Pires" para "Vicente Pires"                     3
Canonizado: Transformado de "Sul (Águas Claras)" para "Águas Claras Sul"         2
Canonizado: Transformado de "Taguatinga/DF" para "Taguatinga"                    2
Canonizado: Transformado de "TAGUATINGA" para "Taguatinga"                       2
Canonizado: Transformado de "Águas Claras/DF" para "Águas Claras"                2
Canonizado: Transformado de "Vicente Pires - DF" para "Vicente Pires"            2
Canonizado: Transformado de "taguatinga sul" para "Taguatinga Sul"               2
Canonizado: Transformado de "S

In [67]:
# ordena e imprime linha por linha
bairros_ordenados = df['bairro_informado'].unique()
bairros_ordenados.sort()

print("Cidades normalizadas \n")
for bairro in bairros_ordenados:
    print(bairro)

Cidades normalizadas 

Guará
Guará I
Guará II
Taguatinga
Taguatinga Sul
Vicente Pires
Águas Claras
Águas Claras Norte
Águas Claras Sul
nan


re.sub(r'\D', '', str(valor)): A expressão regular \D localiza tudo o que não for número (como hifens, pontos ou espaços) e substitui por vazio, deixando apenas os dígitos puros. (atualizado)

.zfill(8): O método zero fill preenche a string com zeros à esquerda até que ela atinja exatamente 8 caracteres. Isso corrige um problema clássico de arquivos CSV/Excel, que costumam "engolir" o zero inicial de CEPs que começam com esse dígito (ex: transforma 7085570 em 07085570). (atualizado)

len(...) != 8 or ... == '00000000' (Validação Local): Regra de negócio que verifica se o CEP reconstruído possui um comprimento válido ou se trata de um preenchimento "preguiçoso" (como oito zeros seguidos). Valores identificados nesta malha fina são catalogados em uma lista de log e forçados a se tornarem pd.NA. (novo)

.apply(normalizar_cep): Executa essa função linha por linha exclusivamente na coluna cep, atualizando os dados do DataFrame de forma definitiva com base nas lógicas de higienização e validação. (atualizado)

In [68]:
# 1. Função para normalizar a data e gerar o log de auditoria
def normalizar_data_auditoria(valor):
    # Se já for nulo nativo ou apenas espaços em branco
    if pd.isna(valor) or str(valor).strip() == '':
        # pd.NaT é o "Not a Time", o nulo oficial do Pandas para campos de data
        return pd.NaT, 'Já era Nulo ou Vazio'
    
    texto_data = str(valor).strip()
    
    try:
        # Verifica se o texto começa com 4 dígitos seguidos de um traço (ex: 2025-11-28)
        # Isso silencia o aviso, pois não usamos dayfirst=True em datas que já estão no padrão ISO
        if re.match(r'^\d{4}-', texto_data):
            data_convertida = pd.to_datetime(texto_data)
        else:
            # Para formatos brasileiros (18/07/2025) ou por extenso (Apr 3), forçamos o dia primeiro
            data_convertida = pd.to_datetime(texto_data, dayfirst=True)
        
        return data_convertida, 'Convertido com sucesso para datetime64 (YYYY-MM-DD)'
        
    except Exception:
        # Se houver algum texto absurdo (ex: "Data Desconhecida", "Não lembro")
        return pd.NaT, f'Inválido: O texto original "{valor}" não é uma data reconhecível'

# 2. Recebendo a tupla em DUAS colunas simultaneamente
# O dado transformado volta para 'data_cadastro' e o log vai para 'auditoria_data_cadastro'
df[['data_cadastro', 'auditoria_data_cadastro']] = df['data_cadastro'].apply(
    lambda x: pd.Series(normalizar_data_auditoria(x))
)

# 3. Contagem rápida para verificar os logs
print("\n--- Resumo da Auditoria de Datas ---")
print(df['auditoria_data_cadastro'].value_counts().to_string())


--- Resumo da Auditoria de Datas ---
auditoria_data_cadastro
Convertido com sucesso para datetime64 (YYYY-MM-DD)    47


In [69]:
# Filtra todas as colunas que NÃO começam com 'auditoria_'
colunas_antigas = [col for col in df.columns if not col.startswith('auditoria_')]

# Mostra o DataFrame apenas com essas colunas
print("tábelas com todas as colunas normalizadas")
display(df[colunas_antigas])

tábelas com todas as colunas normalizadas


,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61 97085-5700,contato@mercadinhobo.com.br,Águas Claras Sul,71926750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,61 95386-2084,contato@comercialsao.com.br,Águas Claras Sul,72140010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,61 92489-1771,contato@armarinhossi.com.br,Taguatinga,72140010,Brasília,DF,2022-02-28,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,61 98433-1053,contato@emporiovalev.com.br,Águas Claras,71929000,Brasília,DF,2023-12-24,Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61 95127-3847,contato@merceariadoi.com.br,Taguatinga,NaN,Brasília,DF,2022-01-08,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82.278.248/9638-34,61 97118-8177,contato@casadastinta.com.br,Águas Claras,NaN,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83.930.103/1051-83,61 99978-8677,contato@distribuidor.com.br,Taguatinga,NaN,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31.165.667/0106-51,61 94139-4116,contato@bazarestrela.com.br,Vicente Pires,72006400,Brasília,DF,2022-08-18,Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10.801.326/7736-02,61 91035-7396,contato@supermercado.com.br,Águas Claras,71926750,Brasília,DF,2023-08-09,Juliana Reis
9,CLI0010,Papelaria Horizonte,34.309.805/0097-88,61 91931-9320,contato@papelariahor.com.br,Águas Claras,71950000,Brasília,DF,2021-03-03,Diego Martins


In [70]:
# 1. Cria uma lista com a coluna de ID e todas as colunas que começam com 'auditoria_'
colunas_auditoria = [col for col in df.columns if col.startswith('auditoria_')]
colunas_selecionadas = ['id_cliente'] + colunas_auditoria

# 2. Imprime um título bonito no terminal
print("\n--- Relatório de Auditoria das Normalizações ---")

# 3. Exibe apenas essas colunas
display(df[colunas_selecionadas])


--- Relatório de Auditoria das Normalizações ---


,id_cliente,auditoria_cep,auditoria_cnpj,auditoria_telefone,auditoria_email,auditoria_bairro,auditoria_data_cadastro
0,CLI0001,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""Sul (Águas Claras...",Convertido com sucesso para datetime64 (YYYY-M...
1,CLI0002,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada (DDI +55 remo...,"Válido: Higienizado (espaços, minúsculas e/ou ...","Canonizado: Transformado de ""Sul (Aguas Claras...",Convertido com sucesso para datetime64 (YYYY-M...
2,CLI0003,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada (DDI +55 remo...,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""Taguatinga - DF"" ...",Convertido com sucesso para datetime64 (YYYY-M...
3,CLI0004,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada,"Válido: Higienizado (espaços, minúsculas e/ou ...","Canonizado: Transformado de ""aguas claras"" par...",Convertido com sucesso para datetime64 (YYYY-M...
4,CLI0005,Inválido: Preenchimento genérico (00000000),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""Taguatinga/DF"" pa...",Convertido com sucesso para datetime64 (YYYY-M...
5,CLI0006,Já era Nulo original,Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada (DDI +55 remo...,Válido: Já estava no padrão correto,Válido: Já estava no padrão correto,Convertido com sucesso para datetime64 (YYYY-M...
6,CLI0007,Já era Nulo original,Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""TAGUATINGA"" para ...",Convertido com sucesso para datetime64 (YYYY-M...
7,CLI0008,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""V. Pires"" para ""V...",Convertido com sucesso para datetime64 (YYYY-M...
8,CLI0009,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada (DDI +55 remo...,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""Aguas claras"" par...",Convertido com sucesso para datetime64 (YYYY-M...
9,CLI0010,Válido: Higienizado (tamanho original correto),Válido: Higienizado e máscara aplicada,Válido: Máscara Celular aplicada,Válido: Já estava no padrão correto,"Canonizado: Transformado de ""Águas Claras/DF"" ...",Convertido com sucesso para datetime64 (YYYY-M...


## Todas as colunas normalizadas, agora somente com os dados nulos, os dados não preenchidos que serão corrigidos com a API de cep

### Justificativas das Normalizações e Tratamento de Dados

Esta etapa do processo de ETL (Extract, Transform, Load) focou na padronização estrutural e semântica da base de dados, visando garantir a qualidade da informação e preparar os dados para ingestão futura em um banco de dados relacional (PostgreSQL).

Abaixo estão as justificativas para cada regra de negócio aplicada:

* **Rastreabilidade e Geração de Histórico de Auditoria:**
* Para garantir total transparência e cumprir os requisitos de governança de dados, o processo de ETL foi projetado para registrar a "linhagem do dado" (*data lineage*). Todas as funções de normalização retornam uma tupla estruturada contendo `(valor_limpo, criterio_aplicado)`. Utilizando o Pandas, esse resultado é desmembrado dinamicamente, gerando colunas auxiliares de histórico (ex: `auditoria_telefone`, `auditoria_cep`) que explicam o que ocorreu com cada célula, preparando a base para alimentar a tabela de Auditoria do banco de dados de forma rastreável linha a linha.


* **Tratamento de Dados Faltantes (Nulos e Vazios):**
* Valores preenchidos apenas com espaços em branco ou strings vazias (`""`) foram convertidos para o nulo nativo do Pandas (`pd.NA` ou `pd.NaT` para datas). Isso corrige a falsa impressão de preenchimento no dataset (como visto no `.info()`) e garante que o banco de dados receba um `NULL` verdadeiro, evitando inconsistências.


* **Ajustes de Tipagem (Data Types):**
* **`id_cliente`:** Mantido como texto (`str`), pois a base possui padrões alfanuméricos (ex: `CLI0016`). A conversão forçada para numérico geraria perda de dados.
* **`data_cadastro`:** Convertida para o tipo nativo `datetime64`. Foram harmonizados múltiplos formatos de origem (ex: `18/07/2025`, `Apr 3, 2025`, `2025-11-28`) utilizando validação via Regex (`^\d{4}-`) para silenciar alertas de formato misto e manter o padrão internacional (`YYYY-MM-DD`). Esta decisão técnica é necessária, pois SGBDs como o PostgreSQL exigem esse formato nativo. Datas absurdas ou incompreensíveis foram convertidas para `pd.NaT` (Not a Time) com o devido registro na auditoria.


* **Padronização de Máscaras (Regex):**
* **`cnpj`:** Mantido como texto para não perder os zeros à esquerda. Aplicou-se uma rotina de higienização removendo pontuações pré-existentes e validando a presença de 14 dígitos. Em seguida, foi injetada a máscara oficial (`XX.XXX.XXX/XXXX-XX`) para uniformidade visual.
* **`telefone`:** Utilizada expressão regular (`re.sub(r'\D', '')`) para extrair apenas dígitos. O código do país (`+55`) foi removido dinamicamente. Os números foram reformatados com base no tamanho da string, recebendo máscaras específicas para celular (11 dígitos) e fixo (10 dígitos). Valores fora desse padrão foram mantidos limpos e sinalizados na auditoria para revisão.


* **Consolidação de Variáveis Categóricas (Bairros):**
* **`bairro_informado`:** A coluna possuía alta variância de digitação (letras maiúsculas/minúsculas e sufixos despadronizados). Foram criadas regras de clusterização (canonização) baseadas em palavras-chave para agrupar as variações em categorias únicas:
* Termos contendo "PIRES" foram consolidados como "Vicente Pires".
* Variações de "Águas Claras", "Guará" e "Taguatinga" foram categorizadas adequadamente, preservando e padronizando os sufixos geográficos ("Sul", "Norte", "I", "II"). A auditoria registra exatamente a transformação (De -> Para).




* **Higienização e Padronização de E-mails:**
* **`email`:** O texto foi convertido inteiramente para minúsculas e os espaços aparados. Em vez de bibliotecas externas, aplicou-se um mapa de tradução nativo (`str.maketrans`) para remover acentuações e cedilha, prevenindo falhas de registro. Adicionalmente, incluiu-se uma malha fina de validação estrutural: e-mails que não continham os caracteres mínimos necessários (`@` e `.`) foram mantidos limpos, porém marcados como "Inválidos/Suspeitos" na auditoria.


* **Higienização e Validação Estrutural de CEPs:**
* **`cep`:** A coluna passou por um processo rigoroso de limpeza (remoção de hifens via Regex) e técnica de *zero fill* (`.zfill(8)`) para restaurar zeros à esquerda perdidos na extração.
* **Validação de Negócio:** CEPs que não atingiram exatamente 8 dígitos numéricos, ou que consistiam em preenchimentos genéricos de formulário (como `'00000000'`), foram classificados como estruturalmente inválidos e convertidos para o nulo nativo (`pd.NA`). Os motivos exatos de aceitação ou reprovação foram registrados na coluna `auditoria_cep`.

# integração com api ViaCEP 

In [71]:
import requests
import time

# 1. Isolar os CEPs únicos (ignorando os nulos) para evitar requisições repetidas
ceps_unicos = df['cep'].dropna().unique()
cache_cep = {}

print(f"A iniciar a consulta no ViaCEP para {len(ceps_unicos)} CEPs únicos...")

# 2. Alimentar o dicionário (Cache em memória) - Tarefas D-02 e E-04
for cep in ceps_unicos:
    cep_str = str(cep)
    url = f"https://viacep.com.br/ws/{cep_str}/json/"
    
    try:
        # timeout=5 evita que o notebook bloqueie caso haja falha de rede
        resposta = requests.get(url, timeout=5)
        
        if resposta.status_code == 200:
            dados = resposta.json()
            
            # O ViaCEP devolve HTTP 200 com {"erro": "true"} para CEPs que não existem
            if dados.get('erro'):
                cache_cep[cep_str] = {'bairro_api': pd.NA, 'status': 'Inválido no ViaCEP'}
            else:
                cache_cep[cep_str] = {'bairro_api': dados.get('bairro', ''), 'status': 'Encontrado'}
        else:
            cache_cep[cep_str] = {'bairro_api': pd.NA, 'status': f'Erro HTTP {resposta.status_code}'}
            
    except requests.exceptions.RequestException:
        cache_cep[cep_str] = {'bairro_api': pd.NA, 'status': 'Timeout/Erro de Rede'}
        
    # Pausa de 100ms para não sobrecarregar o serviço público do ViaCEP
    time.sleep(0.1)

print("Consultas à API concluídas!")

# 3. Aplicar os dados do cache de volta ao DataFrame
def buscar_dados_cache(cep, campo):
    if pd.isna(cep):
        return pd.NA if campo == 'bairro_api' else 'CEP Nulo localmente'
    return cache_cep[str(cep)][campo]

df['bairro_viacep'] = df['cep'].apply(lambda x: buscar_dados_cache(x, 'bairro_api'))
df['status_viacep'] = df['cep'].apply(lambda x: buscar_dados_cache(x, 'status'))

# 4. Regra de Negócio (Tarefa E-05): Comparar bairros e assinalar "precisa_revisao"
def verificar_conflito(linha):
    bairro_local = str(linha['bairro_informado']).upper().strip()
    bairro_api = str(linha['bairro_viacep']).upper().strip()
    status = linha['status_viacep']
    
    # Se o CEP deu erro na API, é nulo, ou não trouxe bairro
    if status != 'Encontrado' or pd.isna(linha['bairro_viacep']) or pd.isna(linha['bairro_informado']):
        return True
        
    # Se os nomes forem diferentes, assinala para revisão
    # (Pode requerer ajuste caso o ViaCEP traga acentos diferentes do seu padrão)
    if bairro_local != bairro_api:
        return True
        
    return False

# Aplica a função linha a linha (axis=1 permite ler várias colunas em simultâneo)
df['precisa_revisao'] = df.apply(verificar_conflito, axis=1)

# 5. Visualizar o resultado das divergências para o relatório (Tarefa D-02)
colunas_conflito = ['id_cliente', 'cep', 'bairro_informado', 'bairro_viacep', 'status_viacep', 'precisa_revisao']
divergencias = df[df['precisa_revisao'] == True][colunas_conflito]

print(f"\n--- Resumo de Revisão (Conflitos ou Erros) ---")
print(f"Total de registos a necessitar de revisão: {len(divergencias)}")
display(divergencias.head(10))

A iniciar a consulta no ViaCEP para 14 CEPs únicos...
Consultas à API concluídas!

--- Resumo de Revisão (Conflitos ou Erros) ---
Total de registos a necessitar de revisão: 47


,id_cliente,cep,bairro_informado,bairro_viacep,status_viacep,precisa_revisao
0,CLI0001,71926750,Águas Claras Sul,Sul (Águas Claras),Encontrado,True
1,CLI0002,72140010,Águas Claras Sul,Taguatinga Norte,Encontrado,True
2,CLI0003,72140010,Taguatinga,Taguatinga Norte,Encontrado,True
3,CLI0004,71929000,Águas Claras,Sul (Águas Claras),Encontrado,True
4,CLI0005,NaN,Taguatinga,NaN,CEP Nulo localmente,True
5,CLI0006,NaN,Águas Claras,NaN,CEP Nulo localmente,True
6,CLI0007,NaN,Taguatinga,NaN,CEP Nulo localmente,True
7,CLI0008,72006400,Vicente Pires,Setor Habitacional Vicente Pires,Encontrado,True
8,CLI0009,71926750,Águas Claras,Sul (Águas Claras),Encontrado,True
9,CLI0010,71950000,Águas Claras,Areal,Encontrado,True


In [72]:
display(divergencias)

,id_cliente,cep,bairro_informado,bairro_viacep,status_viacep,precisa_revisao
0,CLI0001,71926750,Águas Claras Sul,Sul (Águas Claras),Encontrado,True
1,CLI0002,72140010,Águas Claras Sul,Taguatinga Norte,Encontrado,True
2,CLI0003,72140010,Taguatinga,Taguatinga Norte,Encontrado,True
3,CLI0004,71929000,Águas Claras,Sul (Águas Claras),Encontrado,True
4,CLI0005,NaN,Taguatinga,NaN,CEP Nulo localmente,True
5,CLI0006,NaN,Águas Claras,NaN,CEP Nulo localmente,True
6,CLI0007,NaN,Taguatinga,NaN,CEP Nulo localmente,True
7,CLI0008,72006400,Vicente Pires,Setor Habitacional Vicente Pires,Encontrado,True
8,CLI0009,71926750,Águas Claras,Sul (Águas Claras),Encontrado,True
9,CLI0010,71950000,Águas Claras,Areal,Encontrado,True


In [73]:
# 1. CRIAR CÓPIA PARA ANÁLISE (Para não perdermos a comparação original)
df_analise = df.copy()

# 2. Função para classificar o tipo de diferença entre o digitado e o ViaCEP
def classificar_diferenca(linha):
    inf = str(linha['bairro_informado']).upper().strip()
    via = str(linha['bairro_viacep']).upper().strip()
    status = linha['status_viacep']
    
    if status != 'Encontrado' or pd.isna(linha['bairro_viacep']):
        return 'Sem Bairro ViaCEP'
        
    if inf == via:
        return 'Iguais'
        
    # Regra 1: Mesma região, apenas sintaxe diferente (ex: Águas Claras Sul vs Sul (Águas Claras))
    if ('ÁGUAS CLARAS' in inf and 'ÁGUAS CLARAS' in via) and \
       (('SUL' in inf and 'SUL' in via) or ('NORTE' in inf and 'NORTE' in via)):
        return 'Sintaxe Diferente (Equivalentes)'
        
    if 'VICENTE PIRES' in inf and 'VICENTE PIRES' in via:
        return 'Sintaxe Diferente (Equivalentes)'
        
    # Regra 2: Complemento (ex: Taguatinga vs Taguatinga Norte / Águas Claras vs Sul (Águas Claras))
    # Deteta se um nome está contido dentro do outro
    if inf in via or via in inf:
        return 'Complemento (Mais Específico/Genérico)'
        
    # Regra 3: Totalmente Errado/Divergente (ex: Águas Claras vs Areal, Taguatinga Sul vs Taguatinga Norte)
    return 'Divergente / Errado'

# Aplica a classificação na nossa tabela de análise
df_analise['categoria_diferenca'] = df_analise.apply(classificar_diferenca, axis=1)


# 3. IMPRIMIR OS RESUMOS SOLICITADOS
print("--- 1. Resumo do Status do ViaCEP ---")
print(df['status_viacep'].value_counts().to_string())

print("\n--- 2. Resumo da Qualidade dos Bairros Informados ---")
print(df_analise['categoria_diferenca'].value_counts().to_string())

print("\n--- 3. Exemplo dos Bairros Divergentes / Errados ---")
# Mostra apenas os que estavam realmente errados (ex: Areal, troca de Norte/Sul)
divergentes = df_analise[df_analise['categoria_diferenca'] == 'Divergente / Errado']
display(divergentes[['id_cliente', 'bairro_informado', 'bairro_viacep']].head())


# 4. APLICAR AS REGRAS DEFINITIVAS NO DATAFRAME PRINCIPAL (O ETL REAL)

# A) Apagar o CEP (Transformar em NaN) se o status for "Inválido no ViaCEP"
mascara_cep_invalido = df['status_viacep'] == 'Inválido no ViaCEP'
df.loc[mascara_cep_invalido, 'cep'] = pd.NA
# Atualizamos a auditoria do CEP para registar esta decisão final
df.loc[mascara_cep_invalido, 'auditoria_cep'] = 'Inválido: Reprovado na consulta ao ViaCEP (Inexistente)'

# B) Substituir o bairro informado pelo bairro do ViaCEP (sempre que o ViaCEP trouxe resultado)
mascara_tem_viacep = df['bairro_viacep'].notna()
df.loc[mascara_tem_viacep, 'bairro_informado'] = df.loc[mascara_tem_viacep, 'bairro_viacep']

# Limpar as colunas temporárias que usámos do ViaCEP no dataframe principal (já cumpriram a função)
df = df.drop(columns=['bairro_viacep', 'status_viacep', 'precisa_revisao'])

print("\n✔ Atualização concluída: CEPs inválidos apagados e Bairros substituídos com sucesso na base principal!")

--- 1. Resumo do Status do ViaCEP ---
status_viacep
Encontrado             28
CEP Nulo localmente    16
Inválido no ViaCEP      3

--- 2. Resumo da Qualidade dos Bairros Informados ---
categoria_diferenca
Sem Bairro ViaCEP                         19
Complemento (Mais Específico/Genérico)    12
Sintaxe Diferente (Equivalentes)          10
Divergente / Errado                        6

--- 3. Exemplo dos Bairros Divergentes / Errados ---


,id_cliente,bairro_informado,bairro_viacep
1,CLI0002,Águas Claras Sul,Taguatinga Norte
9,CLI0010,Águas Claras,Areal
11,CLI0012,Águas Claras,Areal
20,CLI0021,Taguatinga Sul,Taguatinga Norte
31,CLI0032,Águas Claras,Areal



✔ Atualização concluída: CEPs inválidos apagados e Bairros substituídos com sucesso na base principal!


In [77]:
# 1. Carregamos apenas a coluna original direto do CSV para garantir 100% de fidelidade
# (Ajuste o sep=';' para sep=',' se o seu CSV usar vírgula como separador)
df_original = pd.read_csv('../dados/clientes_distribuidora_vetor.csv', sep=';')

# 2. Montamos o DataFrame temporário de visualização
df_visaofinal = pd.DataFrame()
df_visaofinal['id_cliente'] = df['id_cliente']

# O bairro CRU, exatamente como estava no CSV original
df_visaofinal['bairro_antigo_csv'] = df_original['bairro_informado']

# O bairro que ficou definitivamente na sua base principal (pós-limpeza e pós-ViaCEP)
df_visaofinal['bairro_final'] = df['bairro_informado']

# Verifica se o bairro estava errado (Baseado na nossa classificação ViaCEP)
df_visaofinal['bairro_estava_errado'] = df_analise['categoria_diferenca'].apply(
    lambda x: 'Sim' if x == 'Divergente / Errado' else 'Não'
)

# Verifica se o CEP foi confirmado no ViaCEP
df_visaofinal['cep_confirmado'] = df_analise['status_viacep'].apply(
    lambda x: 'Sim' if x == 'Encontrado' else f'Não ({x})'
)

# 3. Exibe sem limite de linhas
pd.set_option('display.max_rows', None)

print("\n--- Auditoria Final: Bairro Original vs Bairro Definitivo ---")
display(df_visaofinal)

# 4. Restaura o limite padrão para não travar o notebook depois
pd.reset_option('display.max_rows')


--- Auditoria Final: Bairro Original vs Bairro Definitivo ---


,id_cliente,bairro_antigo_csv,bairro_final,bairro_estava_errado,cep_confirmado
0,CLI0001,Sul (Águas Claras),Sul (Águas Claras),Não,Sim
1,CLI0002,Sul (Aguas Claras),Taguatinga Norte,Sim,Sim
2,CLI0003,Taguatinga - DF,Taguatinga Norte,Não,Sim
3,CLI0004,aguas claras,Sul (Águas Claras),Não,Sim
4,CLI0005,Taguatinga/DF,Taguatinga,Não,Não (CEP Nulo localmente)
5,CLI0006,Águas Claras,Águas Claras,Não,Não (CEP Nulo localmente)
6,CLI0007,TAGUATINGA,Taguatinga,Não,Não (CEP Nulo localmente)
7,CLI0008,V. Pires,Setor Habitacional Vicente Pires,Não,Sim
8,CLI0009,Aguas claras,Sul (Águas Claras),Não,Sim
9,CLI0010,Águas Claras/DF,Areal,Sim,Sim
